Import Library

In [ ]:
!pip install pymupdf
import fitz
from google.colab import files

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 MB 47.5 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 141.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [pymupdf]


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Upload File PDF

In [ ]:
print("Silakan pilih file PDF...")
uploaded = files.upload()

if uploaded:
    filename = next(iter(uploaded))
    file_bytes = uploaded[filename]
    print(f"\nFile '{filename}' berhasil di-upload ({len(file_bytes) / 1024:.2f} KB).")
else:
    filename = None
    file_bytes = None
    print("\n Tidak ada file yang di-upload.")

Silakan pilih file PDF...


Saving Ruangguru - Kalimat Efektif.pdf to Ruangguru - Kalimat Efektif.pdf

File 'Ruangguru - Kalimat Efektif.pdf' berhasil di-upload (453.09 KB).


Validasi PDF

In [ ]:
MIN_CHAR_THRESHOLD = 20
is_valid = False
doc = None
full_text = []
halaman_kosong_atau_scan = []

if not file_bytes:
    print("Belum terdapat file pdf yang di-upload.")

elif not filename.lower().endswith(".pdf"):
    print(f"Format tidak valid: '{filename}' bukan file PDF.")

elif len(file_bytes) == 0:
    print(f"File '{filename}' berukuran 0 KB.")

else:
    try:
        doc = fitz.open(stream=file_bytes, filetype="pdf")

        if doc.is_encrypted:
            print(f"File '{filename}' terenkripsi atau terkunci password.")
        elif len(doc) == 0:
            print(f"File '{filename}' memiliki 0 halaman.")
        else:
            total_gambar = 0
            for page_num in range(len(doc)):
                page = doc[page_num]
                text = page.get_text("text").strip()
                total_gambar += len(page.get_images(full=True))

                if len(text) == 0:
                    halaman_kosong_atau_scan.append(page_num + 1)
                full_text.append(text)

            gabungan_teks = "\n".join(full_text).strip()

            if len(gabungan_teks) < MIN_CHAR_THRESHOLD:
                print(" PDF TIDAK MEMENUHI SYARAT EKSTRAKSI TEKS")
                if total_gambar > 0:
                    print("Diagnosa : PDF terdeteksi sebagai hasil SCAN (berisi gambar tanpa layer teks).")
                    print("Solusi   : Gunakan library OCR (pytesseract/EasyOCR) untuk membaca teks.")
                else:
                    print("Diagnosa : PDF KOSONG (tidak ada karakter teks maupun gambar).")
                    print("Solusi   : Pastikan file PDF berisi teks yang dapat diseleksi.")
            else:
                is_valid = True
                print(f"File '{filename}' lolos validasi dan siap diproses.")

    except Exception as e:
        print(f"File corrupt atau bukan PDF valid. Detail error: {e}")

File 'Ruangguru - Kalimat Efektif.pdf' lolos validasi dan siap diproses.


Ringkasan PDF

In [ ]:
if not is_valid:
    print("Informasi tidak dapat ditampilkan karena PDF tidak valid")
else:
    metadata = doc.metadata
    print("="*55)
    print("INFORMASI FILE PDF")
    print("="*55)
    print(f"Nama File      : {filename}")
    print(f"Ukuran File    : {len(file_bytes) / 1024:.2f} KB")
    print(f"Jumlah Halaman : {len(doc)}")

    if halaman_kosong_atau_scan:
        print(f"Peringatan     : Halaman {halaman_kosong_atau_scan} kosong atau berupa gambar scan.")
    print("="*55)

INFORMASI FILE PDF
Nama File      : Ruangguru - Kalimat Efektif.pdf
Ukuran File    : 453.09 KB
Jumlah Halaman : 5


Preview PDF

In [ ]:
if not is_valid:
    print("Ekstraksi tidak dapat dilakukan karena PDF tidak valid.")
else:

    for idx, teks_halaman in enumerate(full_text):
        print(f"\n--- [Halaman {idx + 1}] ---")
        print(teks_halaman if teks_halaman else "[Halaman ini kosong atau berupa gambar scan]")

    hasil_ekstrak = "\n".join(full_text).strip()
    doc.close()


--- [Halaman 1] ---
Daftar Isi 
 Kelengkapan Struktur dan Kelogisan 
 1 
 Kesejajaran 
 3 
 Kehematan dan Kecermatan 
 4 
Modul Kilat ini milik jwooos
(fadiyank121313@gmail.com)
jwooos fadiyank121313@gmail.com
jwooos fadiyank121313@gmail.com
jwooos fadiyank121313@gmail.com

--- [Halaman 2] ---
A 
 Kelengkapan Struktur dan 
 Kelogisan 
 1. 
 Kelengkapan Struktur 
 Sebuah  kalimat  dikatakan  efektif  jika  memiliki 
 struktur  minimal  kalimat  yang  lengkap ,  yakni 
 subjek  dan  predikat . 
 Subjek + Predikat 
 Namun,  jika  kalimatnya  berupa  kalimat  aktif 
 transitif ,  objek  juga  menjadi  unsur  kalimat 
 lengkap  yang  wajib  ada .  Struktur  kalimat  yang 
 tidak  lengkap  dapat  disebabkan  oleh  hal-hal 
 sebagai berikut. 
 a.  Tidak Adanya Subjek 
 Contoh: 
 Dalam rapat itu  membahas 
 K                     P 
 pemilihan ketua RT . 
 O 
 Kalimat  tersebut  tidak  mengandung  subjek 
 karena  adanya  kata  depan  dalam  yang 
 membuatnya menjadi unsur  keterangan . 
 Kali

In [ ]:
import json
pages_payload = [{"page": idx + 1, "text": teks} for idx, teks in enumerate(full_text)]
with open("extracted_pages.json", "w", encoding="utf-8") as f:
    json.dump(pages_payload, f, ensure_ascii=False, indent=2)
files.download("extracted_pages.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>